# AI-Publisher Model Test
ToS uyumlu: direkt HuggingFace inference, Docker/Flask/Ngrok yok, kisa session.

In [ ]:
# === HAZIRLIK: Drive mount + bagimliliklar ===
import os, gc, time, json
from datetime import datetime

from google.colab import drive
drive.mount('/content/drive')

os.environ['HF_HOME'] = '/content/drive/MyDrive/Colab_Cache/huggingface'
os.environ['TORCH_HOME'] = '/content/drive/MyDrive/Colab_Cache/torch'
os.environ['XDG_CACHE_HOME'] = '/content/drive/MyDrive/Colab_Cache'
os.makedirs('/content/drive/MyDrive/Colab_Cache', exist_ok=True)

OUTPUT_DIR = '/content/output'
DRIVE_OUTPUT_DIR = '/content/drive/MyDrive/Colab_Notebooks/test_outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)

results_log = []

!pip install -q diffusers transformers accelerate imageio[ffmpeg] torch opencv-python-headless scipy
import torch; import numpy as np; import imageio

In [ ]:
# === edge-tts kurulumu (Colab test, XTTS RunPod'da) ===
!pip install -q edge-tts

In [ ]:
# === GPU KONTROL ===
print(f'CUDA: {torch.cuda.is_available()}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A"}')
total = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
print(f'VRAM: {total:.1f} GB')

In [ ]:
def write_video(frames, out_path, fps=8):
    import os, numpy as np, cv2
    arrs = []
    for i, f in enumerate(frames):
        arr = np.array(f)
        if arr.dtype != np.uint8:
            arr = (np.clip(arr, 0.0, 1.0) * 255).astype(np.uint8)
        if arr.ndim == 3 and arr.shape[2] == 4:
            arr = arr[:, :, :3]
        arrs.append(arr)
    h, w = arrs[0].shape[:2]
    print(f'[VIDEO] {w}x{h}, {len(arrs)} frames, {fps} fps, {len(arrs)/fps:.1f}s')
    print(f'[VIDEO] frame dtype={arrs[0].dtype} min={arrs[0].min()} max={arrs[0].max()}')
    from PIL import Image
    debug_png = out_path.replace('.mp4', '_frame0.png')
    Image.fromarray(arrs[0]).save(debug_png)
    print(f'[VIDEO] first frame saved: {debug_png}')
    fourcc = cv2.VideoWriter_fourcc(*"MJPG")
    vw = cv2.VideoWriter(out_path, fourcc, fps, (w, h))
    if not vw.isOpened():
        raise RuntimeError('cv2.VideoWriter failed to open')
    for arr in arrs:
        vw.write(cv2.cvtColor(arr, cv2.COLOR_RGB2BGR))
    vw.release()
    size_kb = os.path.getsize(out_path) / 1024
    print(f'[VIDEO] output: {out_path} ({size_kb:.0f} KB)')
    return out_path


---
## Test 1: Gorsel Uretimi (SDXL)

In [ ]:
def test_sdxl():
    from diffusers import StableDiffusionXLPipeline
    print('[LOAD] SDXL yukleniyor...')
    pipe = StableDiffusionXLPipeline.from_pretrained(
        'stabilityai/stable-diffusion-xl-base-1.0',
        torch_dtype=torch.float16, variant='fp16', use_safetensors=True
    )
    pipe.enable_model_cpu_offload()
    pipe.enable_vae_tiling()
    prompt = 'Professional food photography, a steaming ceramic coffee cup on a rustic wooden table, soft golden morning light, warm atmosphere, shallow depth of field focused on the cup, rich textures, ultra-detailed, 8K'
    print(f'[RUN] prompt: {prompt}')
    image = pipe(prompt=prompt, num_inference_steps=25).images[0]
    out = f'{OUTPUT_DIR}/sdxl_test.png'
    image.save(out); display(image)
    del pipe; return [out]

run_test('SDXL (Gorsel)', test_sdxl)

---
## Test 2: Video Uretimi (CogVideoX-2b)

In [ ]:
def test_cogvideo():
    from diffusers import CogVideoXPipeline
    print('[LOAD] CogVideoX-2b yukleniyor...')
    pipe = CogVideoXPipeline.from_pretrained('THUDM/CogVideoX-2b', torch_dtype=torch.float16)
    pipe.enable_model_cpu_offload()
    pipe.vae.enable_tiling()
    prompt = 'Cinematic drone footage of a lighthouse on a rocky cliff at sunset, golden hour lighting, dramatic clouds, waves crashing against rocks, slow camera orbit around the lighthouse, warm orange and blue color grading, 24fps, epic atmosphere, photorealistic'
    print(f'[RUN] prompt: {prompt}')
    output = pipe(prompt=prompt, num_frames=49, num_inference_steps=25, guidance_scale=6)
    frames = output.frames[0]
    display(frames[0])  # first frame preview
    out = f'{OUTPUT_DIR}/cogvideo_test.mp4'
    write_video(frames, out, fps=7)
    from IPython.display import Video
    display(Video(out, width=512))
    del pipe; return [out]

run_test('CogVideoX-2b (Video)', test_cogvideo)

---
## Test 3: Gorselden Video (CogVideoX-2b-I2V)

In [ ]:
def test_i2v():
    init_img = f'{OUTPUT_DIR}/sdxl_test.png'
    if not os.path.exists(init_img):
        print('[SKIP] Test 1 gorseli bulunamadi'); return []
    from diffusers import StableVideoDiffusionPipeline
    from diffusers.utils import load_image
    print('[LOAD] SVD (Stable Video Diffusion) yukleniyor...')
    pipe = StableVideoDiffusionPipeline.from_pretrained('stabilityai/stable-video-diffusion-img2vid', torch_dtype=torch.float16, variant='fp16')
    pipe.enable_model_cpu_offload()
    image = load_image(init_img)
    image = image.resize((1024, 576))
    print('[RUN] I2V: kahve gorselinden video...')
    frames = pipe(image=image, decode_chunk_size=8, num_frames=25).frames[0]
    display(frames[0])  # first frame preview
    out = f'{OUTPUT_DIR}/svd_i2v_test.mp4'
    write_video(frames, out, fps=4)
    from IPython.display import Video
    display(Video(out, width=512))
    del pipe; return [out]

run_test('SVD (Gorselden Video)', test_i2v)

---
## Test 4: Turkce TTS (XTTS)

In [ ]:
def test_tts():
    !pip install -q edge-tts
    import subprocess as sp
    text = 'AI Publisher video uretim sistemine hos geldiniz. Bugun sizlere ozel bir icerik hazirladik.'
    print(f'[RUN] text: {text}')
    out = f'{OUTPUT_DIR}/tts_test.mp3'
    sp.run(['edge-tts', '--text', text, '--voice', 'tr-TR-EmelNeural', '--write-media', out], check=True)
    from IPython.display import Audio
    display(Audio(out))
    return [out]

run_test('Edge-TTS (Turkce Ses)', test_tts)

---
## Test 5: Ses Efekti (AudioLDM2)

In [ ]:
def test_sfx():
    from diffusers import AudioLDM2Pipeline
    print('[LOAD] AudioLDM2 yukleniyor...')
    import json, os
    from huggingface_hub import snapshot_download
    model_path = snapshot_download(repo_id='cvssp/audioldm2')
    # Patch model_index.json AND text_encoder/config.json
    for rel_path in ['model_index.json', os.path.join('text_encoder', 'config.json')]:
        cfg_path = os.path.join(model_path, rel_path)
        if os.path.exists(cfg_path):
            with open(cfg_path) as f:
                cfg = f.read()
            if 'GPT2Model' in cfg:
                with open(cfg_path, 'w') as f:
                    f.write(cfg.replace('GPT2Model', 'GPT2LMHeadModel'))
                print(f'[PATCH] {rel_path}: GPT2Model -> GPT2LMHeadModel')
    pipe = AudioLDM2Pipeline.from_pretrained('cvssp/audioldm2', torch_dtype=torch.float16)
    pipe.enable_model_cpu_offload()
    prompt = 'Gentle ocean waves rhythmically crashing on a sandy beach, seagulls calling in the distance, light breeze rustling palm leaves, peaceful and relaxing coastal atmosphere, high quality ambient field recording'
    print(f'[RUN] prompt: {prompt}')
    audio = pipe(prompt=prompt, audio_length_in_s=6.0, num_inference_steps=20).audios[0]
    from scipy.io.wavfile import write as write_wav
    out = f'{OUTPUT_DIR}/sfx_test.wav'
    write_wav(out, 16000, audio)
    from IPython.display import Audio
    display(Audio(out))
    del pipe; return [out]

run_test('AudioLDM2 (Ses Efekti)', test_sfx)

---
## Final Rapor

In [ ]:
print('=' * 80)
print(f'TEST RAPORU - {datetime.now().strftime("%Y-%m-%d %H:%M")}')
print('=' * 80)
print(f'{"Test":<35} {"Durum":<8} {"Sure":<8} {"VRAM":<10} {"Boyut":<8}')
print('-' * 80)
pass_ = 0; fail_ = 0
for r in results_log:
    s = r['status']
    if s == 'OK': pass_ += 1; status = 'OK'
    else: fail_ += 1; status = 'FAIL'
    print(f'{r["name"]:<35} {status:<8} {r["duration_s"]:<8}s {r["vram_used_gb"]:<10}GB {r["file_size_mb"]:<8}MB')
    if r['error']:
        print(f'  HATA: {r["error"]}')
print('-' * 80)
print(f'Toplam: {len(results_log)} test | {pass_} basarili | {fail_} basarisiz')
print(f'Ciktilar: {DRIVE_OUTPUT_DIR}')

# Raporu kaydet
report = {'test_date': datetime.now().isoformat(), 'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A', 'results': results_log}
with open(f'{OUTPUT_DIR}/test_report.json', 'w') as f:
    json.dump(report, f, indent=2)
import shutil
shutil.copy2(f'{OUTPUT_DIR}/test_report.json', f'{DRIVE_OUTPUT_DIR}/test_report.json')
print(f'Rapor: {DRIVE_OUTPUT_DIR}/test_report.json')
print('=' * 80)